In [1]:
from pathlib import Path
import pandas as pd

# Paths are relative to this notebook (Codes/), so the notebook runs on any machine
DATA_DIR = Path("../Dataset")
RAW_FILE = DATA_DIR / "coffeeshop_dataset.csv"
CLEAN_FILE = DATA_DIR / "coffeeshop_dataset_clean.csv"

## 1. Load and inspect the raw data

In [2]:
raw = pd.read_csv(RAW_FILE)
print(raw.shape)
raw.head()

(3636, 6)


,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 10:15:50.520,card,ANON-0000-0000-0001,38.7,Latte
1,2024-03-01,2024-03-01 12:19:22.539,card,ANON-0000-0000-0002,38.7,Hot Chocolate
2,2024-03-01,2024-03-01 12:20:18.089,card,ANON-0000-0000-0002,38.7,Hot Chocolate
3,2024-03-01,2024-03-01 13:46:33.006,card,ANON-0000-0000-0003,28.9,Americano
4,2024-03-01,2024-03-01 13:48:14.626,card,ANON-0000-0000-0004,38.7,Latte


In [3]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3636 entries, 0 to 3635
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         3636 non-null   object 
 1   datetime     3636 non-null   object 
 2   cash_type    3636 non-null   object 
 3   card         3547 non-null   object 
 4   money        3636 non-null   float64
 5   coffee_name  3636 non-null   object 
dtypes: float64(1), object(5)
memory usage: 170.6+ KB


In [4]:
# Date range covered by the data
dt = pd.to_datetime(raw["datetime"])
print("From:", dt.min(), "| To:", dt.max())
print("Unique days:", raw["date"].nunique())

From: 2024-03-01 10:15:50.520000 | To: 2025-03-23 18:11:38.635000
Unique days: 381


In [5]:
# Price range per product: shows prices changed during the period (checked in analysis later)
raw.groupby("coffee_name")["money"].agg(["count", "min", "max"]).sort_values("count", ascending=False)

,count,min,max
coffee_name,,,
Americano with Milk,824,27.92,35.0
Latte,782,32.82,40.0
Americano,578,23.02,30.0
Cappuccino,501,32.82,40.0
Cortado,292,23.02,30.0
Hot Chocolate,282,32.82,40.0
Cocoa,243,32.82,40.0
Espresso,134,18.12,25.0


## 2. Data quality checks

In [6]:
# Full-row duplicates (all columns identical). Never de-duplicate on a single column:
# that keeps only one row per value and silently drops real sales.
print("Full-row duplicates:", raw.duplicated().sum())
print("Duplicate timestamps:", raw["datetime"].duplicated().sum())

Full-row duplicates: 0
Duplicate timestamps: 0


In [7]:
# Missing card IDs: are they exactly the cash payments?
pd.crosstab(raw["cash_type"], raw["card"].isna(), colnames=["card is missing"])

card is missing,False,True
cash_type,,
card,3547,0
cash,0,89


**Finding:** all 89 missing `card` values are cash payments (cash has no card ID), so the nulls are expected.
They are kept as empty instead of being filled with a text like "Cash": a filled value would look like one
customer with 89 purchases and distort any per-customer metric.

## 3. Clean

In [8]:
df = (
    raw
    .drop(columns=["date"])  # redundant: same info as the datetime column
    .rename(columns={
        "datetime": "transaction_datetime",
        "cash_type": "payment_type",
        "card": "customer_card_id",
        "money": "price",          # one row = one drink, so this is the price of that drink
        "coffee_name": "product",
    })
)

df["transaction_datetime"] = pd.to_datetime(df["transaction_datetime"])
df["payment_type"] = df["payment_type"].str.title()   # card -> Card, cash -> Cash
df["product"] = df["product"].str.strip()

# Derived columns, created once here so Power BI and SQL use identical definitions
df["date"] = df["transaction_datetime"].dt.date
df["month"] = df["transaction_datetime"].dt.to_period("M").astype(str)   # e.g. 2024-03
df["weekday"] = df["transaction_datetime"].dt.day_name()
df["hour"] = df["transaction_datetime"].dt.hour

df.head()

,transaction_datetime,payment_type,customer_card_id,price,product,date,month,weekday,hour
0,2024-03-01 10:15:50.520,Card,ANON-0000-0000-0001,38.7,Latte,2024-03-01,2024-03,Friday,10
1,2024-03-01 12:19:22.539,Card,ANON-0000-0000-0002,38.7,Hot Chocolate,2024-03-01,2024-03,Friday,12
2,2024-03-01 12:20:18.089,Card,ANON-0000-0000-0002,38.7,Hot Chocolate,2024-03-01,2024-03,Friday,12
3,2024-03-01 13:46:33.006,Card,ANON-0000-0000-0003,28.9,Americano,2024-03-01,2024-03,Friday,13
4,2024-03-01 13:48:14.626,Card,ANON-0000-0000-0004,38.7,Latte,2024-03-01,2024-03,Friday,13


## 4. Validate before export

These checks stop the notebook if the clean table does not reconcile with the raw file.
(Version 1 of this project lost 90% of rows in Power BI without anyone noticing; these checks prevent that.)

In [9]:
assert len(df) == len(raw) == 3636, "Row count changed during cleaning"
assert round(df["price"].sum(), 2) == round(raw["money"].sum(), 2) == 115431.58, "Revenue does not reconcile"
assert df.duplicated().sum() == 0, "Unexpected duplicate rows"
assert (df["price"] > 0).all(), "Non-positive price found"
assert df["customer_card_id"].isna().sum() == (df["payment_type"] == "Cash").sum(), "Missing card IDs are not all cash"
assert df.drop(columns="customer_card_id").notna().all().all(), "Unexpected missing values"

print(f"OK: {len(df):,} rows | revenue ${df['price'].sum():,.2f} | "
      f"{df['transaction_datetime'].min():%d %b %Y} to {df['transaction_datetime'].max():%d %b %Y}")

OK: 3,636 rows | revenue $115,431.58 | 01 Mar 2024 to 23 Mar 2025


## 5. Export

In [10]:
df.to_csv(CLEAN_FILE, index=False)
print("Saved:", CLEAN_FILE)

Saved: ..\Dataset\coffeeshop_dataset_clean.csv


## Notes and limitations
- March 2025 is partial (data ends 23 Mar 2025). Compare months using revenue per day, not monthly totals.
- No cost data: margin per product cannot be calculated.
- No transaction ID: basket size and average transaction value cannot be calculated.
- No inventory data: stock-out risk cannot be measured.
- `customer_card_id` is an anonymised card number; cash customers cannot be identified.